# Lab | Data Aggregation and Filtering

In this challenge, we will continue to work with customer data from an insurance company. We will use the dataset called marketing_customer_analysis.csv, which can be found at the following link:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis.csv

This dataset contains information such as customer demographics, policy details, vehicle information, and the customer's response to the last marketing campaign. Our goal is to explore and analyze this data by first performing data cleaning, formatting, and structuring.

1. Create a new DataFrame that only includes customers who:
   - have a **low total_claim_amount** (e.g., below $1,000),
   - have a response "Yes" to the last marketing campaign.

2. Using the original Dataframe, analyze:
   - the average `monthly_premium` and/or customer lifetime value by `policy_type` and `gender` for customers who responded "Yes", and
   - compare these insights to `total_claim_amount` patterns, and discuss which segments appear most profitable or low-risk for the company.

3. Analyze the total number of customers who have policies in each state, and then filter the results to only include states where there are more than 500 customers.

4. Find the maximum, minimum, and median customer lifetime value by education level and gender. Write your conclusions.

In [2]:
import pandas as pd
import numpy as np

url = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis.csv"
df_raw = pd.read_csv(url)


df = df_raw.copy()
df.columns = (
    df.columns.str.strip().str.lower()
      .str.replace(" ", "_", regex=False)
      .str.replace("/", "_", regex=False)
      .str.replace("-", "_", regex=False)
)
                            
df = df.replace(["nan", "NaN", "NAN", "", " "], np.nan)

          
if "effective_to_date" in df.columns:
    df["effective_to_date"] = pd.to_datetime(df["effective_to_date"], errors="coerce")

for c in ["total_claim_amount", "customer_lifetime_value", "monthly_premium_auto"]:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")


df["response"] = df["response"].astype("string").str.strip().str.title()


C:\Users\Kelvin\AppData\Local\Temp\ipykernel_27848\1400724361.py:20: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["effective_to_date"] = pd.to_datetime(df["effective_to_date"], errors="coerce")


In [3]:
df_low_claim_yes = df[
    (df["total_claim_amount"] < 1000) &
    (df["response"] == "Yes")
].copy()

print("Rows in df_low_claim_yes:", df_low_claim_yes.shape[0])
display(df_low_claim_yes.head())


Rows in df_low_claim_yes: 1399


,unnamed:_0,customer,state,customer_lifetime_value,response,coverage,education,effective_to_date,employmentstatus,gender,...,number_of_open_complaints,number_of_policies,policy_type,policy,renew_offer_type,sales_channel,total_claim_amount,vehicle_class,vehicle_size,vehicle_type
3,3,XL78013,Oregon,22332.439460,Yes,Extended,College,2011-01-11,Employed,M,...,0.0,2,Corporate Auto,Corporate L3,Offer2,Branch,484.013411,Four-Door Car,Medsize,A
8,8,FM55990,California,5989.773931,Yes,Premium,College,2011-01-19,Employed,M,...,0.0,1,Personal Auto,Personal L1,Offer2,Branch,739.200000,Sports Car,Medsize,NaN
15,15,CW49887,California,4626.801093,Yes,Basic,Master,2011-01-16,Employed,F,...,0.0,1,Special Auto,Special L1,Offer2,Branch,547.200000,SUV,Medsize,NaN
19,19,NJ54277,California,3746.751625,Yes,Extended,College,2011-02-26,Employed,F,...,1.0,1,Personal Auto,Personal L2,Offer2,Call Center,19.575683,Two-Door Car,Large,A
27,27,MQ68407,Oregon,4376.363592,Yes,Premium,Bachelor,2011-02-28,Employed,F,...,0.0,1,Personal Auto,Personal L3,Offer2,Agent,60.036683,Four-Door Car,Medsize,NaN


In [4]:
df_yes = df[df["response"] == "Yes"].copy()

summary_yes = (
    df_yes.groupby(["policy_type", "gender"])
          .agg(
              customers=("customer", "count"),
              avg_monthly_premium=("monthly_premium_auto", "mean"),
              median_monthly_premium=("monthly_premium_auto", "median"),
              avg_clv=("customer_lifetime_value", "mean"),
              median_clv=("customer_lifetime_value", "median"),
              avg_claim=("total_claim_amount", "mean"),
              median_claim=("total_claim_amount", "median"),
          )
          .reset_index()
          .sort_values(["avg_clv", "avg_claim"], ascending=[False, True])
)

display(summary_yes)


,policy_type,gender,customers,avg_monthly_premium,median_monthly_premium,avg_clv,median_clv,avg_claim,median_claim
2,Personal Auto,F,540,98.998148,93.0,8339.791842,5380.898636,452.965929,424.330166
5,Special Auto,M,32,86.343750,72.5,8247.088702,5394.594227,429.527942,345.600000
1,Corporate Auto,M,154,92.188312,82.0,7944.465414,6047.025160,408.582459,364.800000
0,Corporate Auto,F,169,94.301775,89.0,7712.628736,5318.896640,433.738499,420.041981
4,Special Auto,F,35,92.314286,87.0,7691.584111,5107.163002,453.280164,420.041981
3,Personal Auto,M,536,91.085821,80.0,7448.383281,5926.729379,457.010178,412.800000


In [5]:
summary_yes["clv_to_claim_ratio"] = summary_yes["avg_clv"] / summary_yes["avg_claim"].replace(0, np.nan)
summary_yes = summary_yes.sort_values(["clv_to_claim_ratio", "avg_monthly_premium"], ascending=False)

display(summary_yes)


,policy_type,gender,customers,avg_monthly_premium,median_monthly_premium,avg_clv,median_clv,avg_claim,median_claim,clv_to_claim_ratio
1,Corporate Auto,M,154,92.188312,82.0,7944.465414,6047.025160,408.582459,364.800000,19.443971
5,Special Auto,M,32,86.343750,72.5,8247.088702,5394.594227,429.527942,345.600000,19.200354
2,Personal Auto,F,540,98.998148,93.0,8339.791842,5380.898636,452.965929,424.330166,18.411521
0,Corporate Auto,F,169,94.301775,89.0,7712.628736,5318.896640,433.738499,420.041981,17.781748
4,Special Auto,F,35,92.314286,87.0,7691.584111,5107.163002,453.280164,420.041981,16.968720
3,Personal Auto,M,536,91.085821,80.0,7448.383281,5926.729379,457.010178,412.800000,16.298069


In [6]:
customers_by_state = (
    df.groupby("state")
      .size()
      .reset_index(name="num_customers")
      .sort_values("num_customers", ascending=False)
)

states_over_500 = customers_by_state[customers_by_state["num_customers"] > 500]

display(customers_by_state.head(10))
print("States with > 500 customers:", states_over_500.shape[0])
display(states_over_500)


,state,num_customers
1,California,3552
3,Oregon,2909
0,Arizona,1937
2,Nevada,993
4,Washington,888


States with > 500 customers: 5


,state,num_customers
1,California,3552
3,Oregon,2909
0,Arizona,1937
2,Nevada,993
4,Washington,888


In [7]:
clv_stats = (
    df.groupby(["education", "gender"])["customer_lifetime_value"]
      .agg(min_clv="min", median_clv="median", max_clv="max", customers="count")
      .reset_index()
      .sort_values(["median_clv"], ascending=False)
)

display(clv_stats)


,education,gender,min_clv,median_clv,max_clv,customers
7,High School or Below,M,1940.981221,6286.731006,83325.38119,1556
6,High School or Below,F,2144.921535,6039.553187,55277.44589,1590
3,College,M,1918.119700,6005.847375,61134.68307,1596
8,Master,F,2417.777032,5729.855012,51016.06704,475
0,Bachelor,F,1904.000852,5640.505303,73225.95652,1700
2,College,F,1898.683686,5623.611187,61850.18803,1608
9,Master,M,2272.307310,5579.099207,50568.25912,412
5,Doctor,M,2267.604038,5577.669457,32677.34284,201
1,Bachelor,M,1898.007675,5548.031892,67907.27050,1572
4,Doctor,F,2395.570000,5332.462694,44856.11397,200


## Bonus

5. The marketing team wants to analyze the number of policies sold by state and month. Present the data in a table where the months are arranged as columns and the states are arranged as rows.

6.  Display a new DataFrame that contains the number of policies sold by month, by state, for the top 3 states with the highest number of policies sold.

*Hint:*
- *To accomplish this, you will first need to group the data by state and month, then count the number of policies sold for each group. Afterwards, you will need to sort the data by the count of policies sold in descending order.*
- *Next, you will select the top 3 states with the highest number of policies sold.*
- *Finally, you will create a new DataFrame that contains the number of policies sold by month for each of the top 3 states.*

7. The marketing team wants to analyze the effect of different marketing channels on the customer response rate.

Hint: You can use melt to unpivot the data and create a table that shows the customer response rate (those who responded "Yes") by marketing channel.

External Resources for Data Filtering: https://towardsdatascience.com/filtering-data-frames-in-pandas-b570b1f834b9

In [ ]:
# your code goes here